# HealthBench worst-30% open-Fusion challenge

Can an open-weights Fusion improve on the baseline across the 157 hardest HealthBench
Professional conversations? The Engine owns the native multi-turn Cases, pinned GPT-5.4 grading
Judge, physician-written rubrics, per-item grading, and aggregation. Every Candidate and Judge
model call is billed through the researcher's provider connection.

This is a challenge metric, not an official HealthBench score. It uses an unclipped mean over the
worst-30% subset, so negative scores are meaningful and remain rankable. The protocol uses one
answer sample, no length-adjusted score, a floating OpenRouter GPT-5.4 route, and does not yet
forward the reference Judge's low-reasoning setting.

The full challenge is intentionally disabled below because it requires hundreds of paid calls.
The `limit=1` rehearsal performs one Candidate answer and a handful of Judge calls.

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

sf.connect()

## 1. Verify the complete grading chain

A `limit=1` run takes the first worst-30 Case through native conversation input, Candidate
execution, the official grader prompt, GPT-5.4 verdict parsing, and aggregation. It is a
plumbing rehearsal: its Report carries the `healthbench-worst30` id but a one-Case score, so
treat the number as diagnostic — never as a challenge result.

In [ ]:
rehearsal_candidate = sf.Model(
    "openrouter/deepseek/deepseek-v4-pro",
    params={"max_tokens": 4096},
)
rehearsal_report = sf.evaluate(rehearsal_candidate, benchmark="healthbench-worst30", limit=1)
rehearsal_report

In [ ]:
{
    "score": rehearsal_report.candidates.only.score,
    "metrics": rehearsal_report.candidates.only.metrics,
    "cases": rehearsal_report.candidates.only.cases,
}

## 2. Define the open-Fusion baseline

The members remain prompt-free to preserve Khoa's measured baseline. The explicit synthesizer
`sf.Model` owns the researcher-editable health synthesis prompt; changing it changes the Candidate,
not the benchmark revision.

In [ ]:
HEALTH_SYNTHESIS_PROMPT = (
    "You are synthesising the single best response to a health-related "
    "conversation by combining independent responses from a panel of models. "
    "A physician-authored rubric will grade your output per criterion — "
    "accuracy, completeness, safety, and appropriate hedging all matter; so "
    "does avoiding harmful or false content.\n\n"
    "Procedure:\n"
    "1. Read every panel response carefully.\n"
    "2. Identify what each contributes that the others miss (facts, safety "
    "caveats, follow-up questions, care-seeking guidance).\n"
    "3. Produce ONE unified response to the final user turn that:\n"
    "   - Keeps every clinically correct, relevant point\n"
    "   - Drops anything inaccurate, unsafe, or speculative\n"
    "   - Resolves disagreements by favouring the safer, better-supported claim\n"
    "   - Stays concise — verbosity is not rewarded\n"
    "4. Do not introduce claims no panel member made.\n\n"
    "Output: the unified response only, no preamble, no JSON wrapper."
)

In [ ]:
deepseek = sf.Model("openrouter/deepseek/deepseek-v4-pro")
kimi = sf.Model("openrouter/moonshotai/kimi-k2.6")
qwen = sf.Model("openrouter/qwen/qwen3.6-plus")

open_trio = sf.Fusion(
    [deepseek, kimi, qwen],
    name="open_trio",
    synthesizer=sf.Model(
        "openrouter/deepseek/deepseek-v4-pro",
        prompt=HEALTH_SYNTHESIS_PROMPT,
    ),
)
open_trio

## 3. Run the full challenge deliberately

The complete Evaluation makes 157 × four Candidate calls plus roughly 350 GPT-5.4 Judge calls.
Set the switch only after inspecting the Candidate and confirming the provider connection.

In [ ]:
RUN_EVALUATION = False

In [ ]:
challenge_report = None
if RUN_EVALUATION:
    challenge_report = sf.evaluate(open_trio, benchmark="healthbench-worst30")
challenge_report

## 4. Inspect and export the artifact

A valid full attempt has non-null `score` and complete verdict coverage. Case Results retain the
Candidate answer, rubric checks, Judge evidence, raw replies, and any failures.

In [ ]:
challenge_report.candidates.only if challenge_report is not None else None

In [ ]:
challenge_report.export("healthbench-worst30.json") if challenge_report is not None else None

## 5. Corrective loop on HealthBench

The same `sf.CorrectiveLoop` from notebook 07 and the DRACO notebook — **one changed
`benchmark=` line**. Nothing about the loop knows what HealthBench is; HealthBench simply
declares a check surface, and the loop consumes it.

What `healthbench-pass.v1` means here: a draft **passes** when it earns at least half the
available positive rubric points, measured on the clamped score — so a draft that trips enough
safety penalties to go negative can never pass. Feedback tells a failing draft only *whether* it
omitted a required element or did something the rubric prohibits: this rubric ships no theme or
category vocabulary, and naming the criterion itself would hand the panel the answer key.

> **Spend warning:** every check is a paid GPT-5.4 Judge call. A two-member, two-round loop
> checks up to 4 times per Case *on top of* the members' answers and the canonical Judge passes,
> so the SDK warns with the ceiling before the first request. `limit` keeps this honest while you
> explore.

In [ ]:
RUN_CORRECTIVE = False

corrective = sf.CorrectiveLoop([deepseek, kimi], judge=qwen, max_rounds=2)
corrective_report = (
    sf.evaluate(corrective, benchmark="healthbench-worst30", limit=1) if RUN_CORRECTIVE else None
)
corrective_report if corrective_report is not None else (
    "Corrective loop disabled — set RUN_CORRECTIVE = True to spend on paid checks."
)